# HVAC v19 retrain — direct upload (no Drive, no OTP)

Uses the **241 MB** bundle `hvac_v19xs_tiled_colab.zip`. No Google Drive / no phone OTP.

### Easiest way to load the file (avoids the reCAPTCHA upload glitch)
1. `Runtime -> Change runtime type -> GPU`, then run cells 1-2.
2. Click the **folder icon** on the LEFT sidebar (file browser).
3. **Drag `hvac_v19xs_tiled_colab.zip` from your computer into that panel** (uploads to /content).
4. Run cell 3 (it finds the zip; if you skipped the drag it pops a file picker instead).
5. Run cells 4-5. At the end it downloads `hvac_yolov8s_v19.pt` -> put it in the repo `models/` folder.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE -> Runtime > Change runtime type > GPU, then Run all again')

In [ ]:
!pip -q install ultralytics==8.4.51

In [ ]:
# Drag the zip into the left file panel first (recommended), then run this.
from google.colab import files
import zipfile, os, glob
found = glob.glob('/content/*.zip')
if not found:
    up = files.upload()              # picker fallback if you didn't drag it in
    found = ['/content/' + k for k in up]
zname = found[0]
print('using:', zname, f'({os.path.getsize(zname)/1e6:.0f} MB)')
os.makedirs('/content/hvac', exist_ok=True)
with zipfile.ZipFile(zname) as z:
    z.extractall('/content/hvac')
os.chdir('/content/hvac')
print('ready:', os.getcwd(), sorted(os.listdir('.'))[:8])

In [ ]:
# Train. ~30-45 min on a T4. (Lower --epochs to 30 for an even faster first run.)
!python train_v11.py --data-yaml yolo_dataset_v19xs_tiled/data.yaml \
    --epochs 60 --device 0 --batch 16 --imgsz 640 --run-name v19

In [ ]:
import glob
from google.colab import files
cands = glob.glob('models/hvac_yolov8s_v19.pt') + glob.glob('runs/**/weights/best.pt', recursive=True)
assert cands, 'No weights found - check the training cell output.'
print('downloading', cands[0])
files.download(cands[0])